In [ ]:
pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import warnings
warnings.filterwarnings('ignore')

print("Import xong!")


In [ ]:
import glob

csv_files = glob.glob('*.csv')
print(f"Tìm thấy {len(csv_files)} file:")
for f in csv_files:
    print(f)

df = pd.concat([pd.read_csv(f, encoding='utf-8', low_memory=False) 
                for f in csv_files], ignore_index=True)

print(f"\nTổng số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]}")


In [ ]:
# Xóa khoảng trắng tên cột
df.columns = df.columns.str.strip()

# Thay inf và NaN bằng median
df.replace([np.inf, -np.inf], np.nan, inplace=True)
for col in df.select_dtypes(include=[np.number]).columns:
    df[col].fillna(df[col].median(), inplace=True)

# Xóa cột zero-variance
before = df.shape[1]
df = df.loc[:, df.nunique() > 1]
print(f"Xóa {before - df.shape[1]} cột zero-variance")

# Xóa dòng duplicate
before = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Xóa {before - df.shape[0]:,} dòng duplicate")

print(f"\nSau cleaning: {df.shape[0]:,} dòng, {df.shape[1]} cột")


In [ ]:
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')

for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')

print(f"Memory sau tối ưu: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


In [ ]:
plt.figure(figsize=(12, 5))
df['Label'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Phân phối các loại traffic')
plt.xlabel('Label')
plt.ylabel('Số lượng')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('label_distribution.png')
plt.show()

print("\nPhân phối nhãn:")
print(df['Label'].value_counts())


In [ ]:
print(df.columns.tolist())

In [ ]:
selected_features = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Mean',
    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s',
    'Packet Length Mean', 'Packet Length Std', 'SYN Flag Count',
    'ACK Flag Count', 'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'URG Flag Count'
]

plt.figure(figsize=(14, 10))
sns.heatmap(df[selected_features].corr(), cmap='coolwarm', annot=False)
plt.title('Correlation Heatmap - 18 Features')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.show()
print("Heatmap xong!")


In [ ]:
# Encode label
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
print("Classes:", list(le.classes_))
print("Encoded:", list(range(len(le.classes_))))

# Chọn 18 feature (tên đúng với dataset)
selected_features = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Mean',
    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s',
    'Packet Length Mean', 'Packet Length Std', 'SYN Flag Count',
    'ACK Flag Count', 'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'URG Flag Count'
]

X = df[selected_features]
y = df['Label']

# Tách train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nX_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")


In [ ]:
# Fix NaN
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

print("NaN sau fix:", np.isnan(X_train).sum())

# SMOTE - dùng 'not majority' thay vì float
smote = SMOTE(sampling_strategy='not majority', random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# UnderSampler
rus = RandomUnderSampler(random_state=42)
X_train, y_train = rus.fit_resample(X_train, y_train)

print("\nPhân phối sau balance:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(le.inverse_transform(unique), counts):
    print(f"  {u}: {c:,}")

print(f"\nX_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print("\nSẵn sàng train model!")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import time

# Lấy mẫu nhỏ để train nhanh hơn
from sklearn.utils import resample
sample_size = 200000
idx = resample(range(len(X_train)), n_samples=sample_size, random_state=42)
X_sample = X_train[idx]
y_sample = y_train[idx]

print(f"Train trên {sample_size:,} mẫu")

# Định nghĩa các model
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes':         GaussianNB(),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

results = {}

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    start = time.time()
    model.fit(X_sample, y_sample)
    elapsed = time.time() - start
    
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, 
                                   target_names=le.classes_, 
                                   output_dict=True)
    results[name] = report
    print(f"Thời gian: {elapsed:.1f}s")
    print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Predict lại bằng Random Forest
rf_model = models['Random Forest']
y_pred_rf = rf_model.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix_rf.png')
plt.show()
print("Confusion Matrix xong!")


In [ ]:
# Bảng so sánh
summary = []
for name, report in results.items():
    summary.append({
        'Model': name,
        'Accuracy': f"{report['accuracy']:.4f}",
        'Precision': f"{report['weighted avg']['precision']:.4f}",
        'Recall': f"{report['weighted avg']['recall']:.4f}",
        'F1-score': f"{report['weighted avg']['f1-score']:.4f}",
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))


In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import datetime

# Tạo lại scaler nếu chưa có
try:
    scaler
except NameError:
    scaler = StandardScaler()
    scaler.fit(X_train)
    print("Đã tạo lại scaler!")

# Lưu model
joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("Đã lưu model!")

In [ ]:
reqs = """pandas
numpy
matplotlib
seaborn
scikit-learn
imbalanced-learn
joblib
"""
with open('requirements.txt', 'w') as f:
    f.write(reqs)
print("Đã tạo requirements.txt!")